In [1]:
import os
import pandas as pd
import numpy as np 
import seaborn as sns
from tqdm import tqdm

from pydub import AudioSegment
from pydub.utils import make_chunks
from IPython.display import Audio

import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit

from tqdm import tqdm

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

In [2]:
def convert_to_mono(audio):
    """Convert stereo audio to mono."""
    return audio.set_channels(1)

def get_split_unique(path, base_filename):
    """
    Splits an audio file into 1.5-second chunks and names each chunk uniquely.

    Args:
        path (str): The file path to the input audio file.
        base_filename (str): The base name to use for naming the chunk files.

    Returns:
        list: A list of in-memory audio chunks.
    """
    # Carga el archivo de audio
    try:
        audio = AudioSegment.from_file(path, "mp3")
    except Exception as e:
        print(f"Error al cargar el archivo de audio {path}: {e}")
        os.remove(path)
        return []
    
    mono_audio = convert_to_mono(audio)
    chunk_length_ms = 1500  
    chunks = make_chunks(mono_audio, chunk_length_ms)

    # Ensure that the chunks directory exists
    if not os.path.exists("chunks"):
        os.makedirs("chunks")

    paths = []
    for i, chunk in enumerate(tqdm(chunks, desc="Processing chunks")):
        chunk_name = f"chunks/{base_filename}_chunk{i}.mp3"  # Unique naming
        
        # Check if the chunk file already exists
        if not os.path.exists(chunk_name):
            chunk.export(chunk_name, format="mp3")
            paths.append(chunk_name)
        else:
            #print(f"Chunk {chunk_name} already exists, skipping.")
            paths.append(chunk_name)

    return paths

def get_split_with_label(row):
    """
    Splits an audio file into chunks and associates each chunk with the audio's original label.

    Args:
        row (pd.Series): A row from a DataFrame containing 'audio_path' and 'label'.

    Returns:
        list of tuples: A list of tuples, where each tuple contains the path to a chunk and the label.
    """
    path = row['audio_path']
    label = row['label']
    base_filename = os.path.splitext(os.path.basename(path))[0]
    chunk_paths = get_split_unique(path, base_filename)
    return [(chunk_path, label) for chunk_path in chunk_paths]

In [3]:
#df = pd.read_csv('/Users/camcortes/Documents/birds-sounds/data/train_specie.csv')
df = pd.read_csv('/Users/camcortes/Documents/birds-sounds/data/test_muestra.csv')
df = df[['audio_path', 'label']]
print(df.shape)
df.shape

(282, 2)


(282, 2)

In [4]:
chunk_data = df.apply(get_split_with_label, axis=1)
chunk_data = [item for sublist in chunk_data for item in sublist]
chunk_df = pd.DataFrame(chunk_data, columns=['audio_path', 'label'])

Processing chunks: 100%|██████████| 43/43 [00:00<00:00, 7223.16it/s]


In [5]:
AudioSegment.from_file("chunks/56374_chunk2.mp3", "mp3")

In [6]:
print(chunk_df.shape)
chunk_df.sample(4)

(8722, 2)


,audio_path,label
4681,chunks/911076_chunk10.mp3,Ochthoeca cinnamomeiventris
7810,chunks/230548_chunk21.mp3,Taraba major
2839,chunks/582208_chunk10.mp3,Spinus psaltria
1297,chunks/439178_chunk25.mp3,Microbates cinereiventris


In [7]:
chunk_df['label'].value_counts()

label
Vireo griseus                 466
Pitangus sulphuratus          315
Cantorchilus nigricapillus    217
Legatus leucophaius           208
Vireo olivaceus               197
                             ... 
Camptostoma obsoletum          11
Myiopagis caniceps             10
Dendroma rufa                   9
Platyrinchus mystaceus          5
Anairetes parulus               4
Name: count, Length: 133, dtype: int64

In [8]:
# filtrar los labels mayores a 1
labels = chunk_df['label'].value_counts()
labels = labels[labels > 1].index
chunk_df = chunk_df[chunk_df['label'].isin(labels)]
chunk_df['label'].value_counts()

label
Vireo griseus                 466
Pitangus sulphuratus          315
Cantorchilus nigricapillus    217
Legatus leucophaius           208
Vireo olivaceus               197
                             ... 
Camptostoma obsoletum          11
Myiopagis caniceps             10
Dendroma rufa                   9
Platyrinchus mystaceus          5
Anairetes parulus               4
Name: count, Length: 133, dtype: int64

In [ ]:
chunk_df.to_csv('/Users/camcortes/Documents/birds-sounds/data/chunks_specie_sample_test.csv', index=False)